# CVD 无障碍图表：自制 Paired Label + 图像到图像（Image-to-Image）模型

**Group 41 · 11-785 Final Project Pipeline v2**

这个 notebook 把项目从 *调色板级别（10 个颜色 → 10 个颜色）* 升级到 *像素级别（整张图 → 整张图）*。

## 整体思路

FigureQA / ChartQA 没有"原图 → 无障碍图"的配对数据。但 **FigureQA 的 `annotations.json` 里存了每张图的原始数据**（每条线的 x/y、每根柱子的数值和颜色）。所以我们可以用 matplotlib **把同一张图重新画两遍**：

```
                ┌─────────────────────────────┐
FigureQA 数据 ─▶│ ① 画原图（原配色/难区分配色） │──▶ input.png  +  input_mask.png
(或随机生成)    └─────────────────────────────┘
       │        ┌─────────────────────────────┐
       └──────▶│ ② 对配色做梯度下降（复用你们的 │──▶ 每种 CVD 一组最优颜色
                │    3 项 loss，不训练网络）      │
                └─────────────────────────────┘
                ┌─────────────────────────────┐
                │ ③ 规则决定是否加 marker /     │──▶ cues
                │    线型 / 柱状纹理            │
                └─────────────────────────────┘
                ┌─────────────────────────────┐
                │ ④ 用同一份数据画 target 图     │──▶ {cvd}.png + {cvd}_mask.png
                └─────────────────────────────┘

训练： (input.png, CVD 类型) ──U-Net──▶ 输出图   ≈ target.png
```

## 每一节做什么

| 节 | 内容 | 需要 GPU? |
|---|---|---|
| 0 | Colab 环境 & 数据路径 | 否 |
| 1 | Imports & 全局配置 | 否 |
| 2 | CVD 模拟 + 色彩工具（复用旧 notebook，并加了 NumPy 版本） | 否 |
| 3 | 3 项 loss（复用旧 notebook，原封不动） | 否 |
| 4 | **Step 0** 读取 FigureQA 标注 → 统一的 chart spec | 否 |
| 5 | **Step 0'** 随机生成合成图表 spec（数据增广，也是没有 FigureQA 时的后备） | 否 |
| 6 | **Step ①** 输入配色：原配色 or "难区分"配色 | 否 |
| 7 | **渲染器**：同一份 spec → RGB 图 / 像素对齐的系列 mask | 否 |
| 8 | **Step ②** 每张图 × 每种 CVD 的配色优化（批量） | 有更快 |
| 9 | **Step ③** 规则添加 marker / 线型 / 纹理 | 否 |
| 10 | **Step ④** 生成并保存整个配对数据集 | 否 |
| 11 | 检查：可视化几组样本 | 否 |
| 12 | 检查：target 质量统计（ΔS、颜色改变量） | 否 |
| 13 | PyTorch Dataset | – |
| 14 | 模型：CVD 条件 U-Net（FiLM） | – |
| 15 | 图像级 loss（L1 + 像素级无障碍 loss + 非系列区域保持） | – |
| 16 | 训练（固定的验证集，按验证 loss 选 checkpoint） | **是** |
| 17 | 测试集评估 + 固定调色板 baseline（Hungarian 匹配） | 是 |
| 18 | 定性对比图 | 是 |
| 19 | 在真实图表（ChartQA）上推理：泛化测试 | 是 |
| 20 | 消融实验 | **是** |
| 21 | 写报告时要注意的点 | – |

**第一次运行建议**：把第 1 节里的 `N_FIGUREQA = 50`、`N_SYNTH = 50`、`EPOCHS = 1` 当 smoke test，跑通后再调大。

## 0. Colab 环境 & 数据路径

和旧 notebook 一样：上传 `figureqa-sample-train-v1.tar.gz` 并解压。**我们只用它的 `annotations.json`（原始数据），不用它的 PNG**——图片由我们自己重新渲染。

如果解压失败或没有 FigureQA，第 10 节会自动只用随机合成图表，整个流程照样能跑。

In [ ]:
# ---- 只在 Colab 里需要：上传并解压 FigureQA sample ----
import os, tarfile

ARCHIVE = "/content/figureqa-sample-train-v1.tar.gz"
if os.path.exists(ARCHIVE):
    with tarfile.open(ARCHIVE, "r:gz") as tar:
        tar.extractall(path="/content/figureqa-sample-train-v1")
    print("解压完成")
else:
    print("没找到压缩包：如果需要，先运行下面这行上传；否则会只用合成数据")
    # from google.colab import files; files.upload()

# 解压后的目录结构可能是 /content/figureqa-sample-train-v1/sample_train1/{png, annotations.json}
for root, dirs, files_ in os.walk("/content/figureqa-sample-train-v1"):
    if "annotations.json" in files_:
        print("找到 annotations.json:", root)

## 1. Imports & 全局配置

所有可调参数都集中在这里。最重要的几个：

- `N_FIGUREQA` / `N_SYNTH`：用多少张 FigureQA 图 + 多少张随机合成图
- `HARD_PALETTE_FRAC`：多大比例的输入图故意换成"在某种色盲下难区分"的配色（让训练集里有足够多需要修复的样本）
- `LAM_TARGET`：生成 target 时 3 项 loss 的权重。**它决定 target 改多少颜色**，第 12 节会打印统计，帮你调
- `CUE_THRESHOLD`：配色优化后，如果最难区分的一对颜色在 CVD 模拟下 ΔE76 仍低于这个值，就加 marker / 线型 / 纹理
- `IMG_SIZE`：图片边长。256 训练快；GPU 够的话 512 文字更清楚

In [ ]:
import os, glob, json, math, time, random, itertools, collections
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.figure import Figure                       # 直接用 Figure 对象渲染，不经过 pyplot，不会占内存/弹出图
from matplotlib.backends.backend_agg import FigureCanvasAgg
from PIL import Image
from scipy.optimize import linear_sum_assignment
from skimage.color import rgb2lab, deltaE_ciede2000, deltaE_cie76
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display

# ---------------- 路径 ----------------
DATA_ROOT = "/content/figureqa-sample-train-v1/sample_train1"   # 含 annotations.json 的目录
OUT_DIR   = "/content/cvd_pairs"                                 # 配对数据集输出目录
REAL_DIR  = "/content/chartqa_test_png"                          # (第 19 节) 放几张 ChartQA 真实图表 PNG

# ---------------- 数据生成 ----------------
SEED = 0
IMG_SIZE = 256              # 渲染尺寸 (正方形)
K_MAX = 10                  # 每张图最多几个系列
N_FIGUREQA = 1000           # 用多少张 FigureQA 图 (0 = 全部)
N_SYNTH = 1000              # 额外随机生成多少张图
HARD_PALETTE_FRAC = 0.5     # 输入图中"难区分配色"的比例
RENAME_COLOR_LABELS = True  # FigureQA 的图例名字就是颜色名 ("Dark Orange")，换色后会误导 → 改成 "Series 1" 等

# ---------------- target 生成 (Step ②③) ----------------
OPT_STEPS = 300             # 每张图配色优化的步数
OPT_LR = 0.05               # 在 logit 空间里的学习率
LAM_TARGET = (1.0, 0.5, 0.5)  # (accessibility, structure, visual) —— 和旧 notebook 默认一致；第 12 节检查后再调
CUE_THRESHOLD = 20.0        # 优化后最小 ΔE76 (CVD 模拟下) 低于它 → 加 marker/线型/纹理
USE_CUES = True             # False = 只改颜色的 target (消融实验用，记得换 OUT_DIR)

# ---------------- 训练 (第 16 节) ----------------
EPOCHS = 20                 # smoke test 用 1
BATCH_SIZE = 16

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| matplotlib", matplotlib.__version__, "| device:", DEVICE)

## 2. CVD 模拟 + 色彩工具

**上半部分直接复制自旧 notebook 第 2 节**（Machado 2009 矩阵，可微分的 PyTorch 版本，用在 loss 里）。

**下半部分是新增的 NumPy 版本**：数据生成和评估时不需要梯度，用 NumPy 更方便。`s_color` 就是报告里的 $S_{color} = \min_{i \neq j} \Delta E(T_{CVD}(c_i), T_{CVD}(c_j))$。

In [ ]:
CVD_TYPES = ["protanopia", "deuteranopia", "tritanopia", "monochromacy"]
CVD_INDEX = {n: i for i, n in enumerate(CVD_TYPES)}
# Broad proposal categories -> specific simulation conditions
BROAD_TO_SPECIFIC = {
    "red-green": ["protanopia", "deuteranopia"],
    "blue-yellow": ["tritanopia"],
    "complete": ["monochromacy"],
}

_LUM = [0.2126, 0.7152, 0.0722]
_MATS = torch.tensor([
    # protanopia (Machado 2009, severity 1.0)
    [[0.152286, 1.052583, -0.204868],
     [0.114503, 0.786281, 0.099216],
     [-0.003882, -0.048116, 1.051998]],
    # deuteranopia
    [[0.367322, 0.860646, -0.227968],
     [0.280085, 0.672501, 0.047413],
     [-0.011820, 0.042940, 0.968881]],
    # tritanopia
    [[1.255528, -0.076749, -0.178779],
     [-0.078411, 0.930809, 0.147602],
     [0.004733, 0.691367, 0.303900]],
    # monochromacy: every channel = luminance
    [_LUM, _LUM, _LUM],
], dtype=torch.float32)

_RGB2XYZ = torch.tensor([[0.4124564, 0.3575761, 0.1804375],
                         [0.2126729, 0.7151522, 0.0721750],
                         [0.0193339, 0.1191920, 0.9503041]], dtype=torch.float32)
_D65 = torch.tensor([0.95047, 1.0, 1.08883], dtype=torch.float32)


def srgb_to_linear(c):
    c = c.clamp(0.0, 1.0)
    return torch.where(c <= 0.04045, c / 12.92,
                       ((c + 0.055) / 1.055).clamp(min=1e-8) ** 2.4)


def linear_to_srgb(c):
    c = c.clamp(0.0, 1.0)
    return torch.where(c <= 0.0031308, 12.92 * c,
                       1.055 * c.clamp(min=1e-8) ** (1 / 2.4) - 0.055)


def simulate_cvd(rgb, cvd_idx):
    """rgb: (B, N, 3) sRGB in [0,1]; cvd_idx: (B,) long or int. Returns (B, N, 3) sRGB."""
    if not torch.is_tensor(cvd_idx):
        cvd_idx = torch.full((rgb.shape[0],), int(cvd_idx), dtype=torch.long, device=rgb.device)
    M = _MATS.to(rgb.device)[cvd_idx]                 # (B,3,3)
    lin = srgb_to_linear(rgb)
    sim = torch.einsum("bij,bnj->bni", M, lin).clamp(0.0, 1.0)
    return linear_to_srgb(sim)


def rgb_to_lab(rgb):
    """rgb (..., 3) sRGB in [0,1] -> CIE L*a*b* (D65)."""
    lin = srgb_to_linear(rgb)
    xyz = lin @ _RGB2XYZ.to(rgb.device).T
    t = xyz / _D65.to(rgb.device)
    eps = 216 / 24389
    f = torch.where(t > eps, t.clamp(min=1e-8) ** (1 / 3), (24389 / 27 * t + 16) / 116)
    L = 116 * f[..., 1] - 16
    a = 500 * (f[..., 0] - f[..., 1])
    b = 200 * (f[..., 1] - f[..., 2])
    return torch.stack([L, a, b], dim=-1)


def pairwise_delta_e(lab):
    """CIE76 pairwise distances. lab (B, N, 3) -> (B, N, N)."""
    diff = lab[:, :, None, :] - lab[:, None, :, :]
    return torch.sqrt((diff ** 2).sum(-1) + 1e-6)


# =================== 以下为新增：NumPy 版本 (数据生成 / 评估用，不需要梯度) ===================
_MATS_NP = _MATS.numpy()


def simulate_cvd_np(rgb, cvd_idx):
    # rgb: 任意形状 (..., 3) 的 sRGB [0,1]，可以是一组颜色，也可以是整张图片 (H,W,3)
    rgb = np.clip(np.asarray(rgb, np.float32), 0, 1)
    lin = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    sim = np.clip(lin @ _MATS_NP[cvd_idx].T, 0, 1)            # 和 torch 版本的 einsum 等价
    return np.where(sim <= 0.0031308, 12.92 * sim, 1.055 * np.power(sim, 1 / 2.4) - 0.055).astype(np.float32)


def lab_np(rgb):
    rgb = np.asarray(rgb, np.float32)
    return rgb2lab(rgb.reshape(-1, 1, 3)).reshape(rgb.shape)


def s_color(colors, cvd_idx=None, metric="cie76"):
    # 图表级无障碍分数：(CVD 模拟后) 最难区分的那一对颜色的 ΔE。cvd_idx=None 表示正常视觉
    c = np.asarray(colors, np.float32)
    if len(c) < 2:
        return float("nan")
    if cvd_idx is not None:
        c = simulate_cvd_np(c, cvd_idx)
    lab = lab_np(c)
    fn = deltaE_ciede2000 if metric == "ciede2000" else deltaE_cie76
    return min(float(fn(lab[i], lab[j])) for i, j in itertools.combinations(range(len(lab)), 2))


def palette_change(old, new):
    # 正常视觉下平均每个颜色改变了多少 (CIE76)，即报告里的 F
    return float(np.mean(deltaE_cie76(lab_np(old), lab_np(new))))


# 自检：NumPy 版本和 torch 版本的模拟结果一致
_x = np.random.rand(5, 3).astype(np.float32)
for _v in range(4):
    _a = simulate_cvd(torch.tensor(_x)[None], _v)[0].numpy()
    assert np.allclose(_a, simulate_cvd_np(_x, _v), atol=1e-4), _v
print("NumPy / torch CVD simulation match ✓")

## 3. 3 项 loss（直接复用旧 notebook 第 6 节，一字未改）

$L = \lambda_1 L_{accessibility} + \lambda_2 L_{structure} + \lambda_3 L_{visual}$

在这个新 pipeline 里它们被用了 **两次**：
1. **第 8 节（生成 target）**：直接对每张图的颜色做梯度下降 → 得到最优配色（不训练网络）
2. **第 15 节（训练 U-Net）**：`accessibility_loss` 作用在 U-Net 输出图像里每个系列的平均颜色上

In [ ]:
def accessibility_loss(bg, new, mask, cvd, margin=30.0, bg_margin=20.0, tau=2.0):
    """Hinge on pairwise CVD-simulated distance among series colors (+ vs background),
    plus a soft-min term on the least distinguishable pair (the S_color objective)."""
    B, K, _ = new.shape
    allc = torch.cat([bg[:, None], new], 1)                              # (B,K+1,3)
    valid = torch.cat([torch.ones_like(mask[:, :1]), mask], 1)
    d = pairwise_delta_e(rgb_to_lab(simulate_cvd(allc, cvd)))            # (B,K+1,K+1)
    pm = valid[:, :, None] & valid[:, None, :]
    pm = pm & ~torch.eye(K + 1, dtype=torch.bool, device=d.device)[None]
    is_bg = torch.zeros(K + 1, dtype=torch.bool, device=d.device); is_bg[0] = True
    bgpair = is_bg[:, None] | is_bg[None, :]
    m = torch.where(bgpair, torch.tensor(bg_margin, device=d.device),
                    torch.tensor(margin, device=d.device))[None]
    hinge = torch.relu(m - d) / margin
    mean_hinge = (hinge * pm).sum((1, 2)) / pm.sum((1, 2)).clamp(min=1)
    # soft-min over series-series pairs only
    sp = pm & ~bgpair[None]
    dd = torch.where(sp, d, torch.full_like(d, 1e4))
    softmin = -tau * torch.logsumexp((-dd / tau).flatten(1), 1)
    has_pairs = sp.flatten(1).any(1)
    smin_term = torch.where(has_pairs, torch.relu(margin - softmin) / margin, torch.zeros_like(softmin))
    return (mean_hinge + 0.5 * smin_term).mean()


def structure_loss(bg, old, new, mask, order_gap=5.0):
    """Keep chart legible/structured under normal vision: (a) contrast of each series vs the
    background is preserved, (b) lightness ordering of series is preserved (ordinal readings)."""
    lo, ln, lb = rgb_to_lab(old), rgb_to_lab(new), rgb_to_lab(bg)[:, None]
    contrast = (torch.norm(ln - lb, dim=-1) - torch.norm(lo - lb, dim=-1)).abs() / 100.0
    contrast = (contrast * mask).sum(1) / mask.sum(1).clamp(min=1)
    Lo, Ln = lo[..., 0], ln[..., 0]
    pm = mask[:, :, None] & mask[:, None, :]
    ordered = (Lo[:, :, None] - Lo[:, None, :]) < -order_gap                 # i darker than j
    viol = torch.relu(Ln[:, :, None] - Ln[:, None, :] + 2.0) / 100.0         # new i not darker
    sel = ordered & pm
    order = (viol * sel).sum((1, 2)) / sel.sum((1, 2)).clamp(min=1)
    return (contrast + order).mean()


def visual_loss(old, new, mask):
    """Stay perceptually close to the original colors (normal-vision CIE76 shift)."""
    d = torch.norm(rgb_to_lab(old) - rgb_to_lab(new), dim=-1) / 100.0
    return ((d * mask).sum(1) / mask.sum(1).clamp(min=1)).mean()


def total_loss(bg, old, new, mask, cvd, lam=(1.0, 0.5, 0.5), **kw):
    la = accessibility_loss(bg, new, mask, cvd, **kw)
    ls = structure_loss(bg, old, new, mask)
    lv = visual_loss(old, new, mask)
    return lam[0] * la + lam[1] * ls + lam[2] * lv, dict(acc=la.item(), struct=ls.item(), vis=lv.item())

## 4. Step 0：读取 FigureQA 标注 → 统一的 chart spec

我们把每张图转成一个 **spec**（一个 dict），之后渲染器只认这个格式：

```python
# 折线图 / 散点图
{"type": "line" | "dot_line" | "scatter", "title": ..., "xlabel": ..., "ylabel": ...,
 "series": [{"name": "Series 1", "x": [...], "y": [...], "color": np.array([r,g,b])}, ...]}
# 柱状图（每根柱子算一个"系列"，因为 FigureQA 每根柱子颜色不同）
{"type": "vbar" | "hbar", ...,
 "series": [{"name": "Cat 1", "value": 42.0, "color": np.array([r,g,b])}, ...]}
```

**先运行下面第一个格子看一下 annotation 的真实字段**。解析函数已经写得比较宽松（兼容几种常见格式），但如果你们的版本字段名不同，改 `figureqa_to_spec` 就行。饼图暂时跳过（旧 notebook 也是这样）。

In [ ]:
# ---- 先看一眼 annotation 长什么样，确认字段名 ----
ann_path = os.path.join(DATA_ROOT, "annotations.json")
if os.path.exists(ann_path):
    _ann = json.load(open(ann_path))
    print(len(_ann), "annotations | 类型分布:", collections.Counter(a["type"] for a in _ann))
    for t in ["line", "dot_line", "vbar_categorical", "hbar_categorical"]:
        ex = next((a for a in _ann if a["type"] == t), None)
        if ex:
            m0 = ex["models"][0]
            print(f"\n[{t}] 顶层 keys: {list(ex.keys())}")
            print(f"  models 数量: {len(ex['models'])} | models[0] keys: {list(m0.keys())}")
            print("  models[0] 示例:", {k: (v[:4] if isinstance(v, list) else v) for k, v in m0.items() if k != 'bboxes'})
else:
    print("没有 annotations.json → 之后只用合成数据")

In [ ]:
def to_rgb01(c):
    # '#aabbcc' / 'aabbcc' / 颜色名 → np.array([r,g,b]) in [0,1]
    if isinstance(c, (list, tuple, np.ndarray)):
        return np.asarray(c, np.float32)[:3]
    for cand in (c, "#" + c, c.replace(" ", "").lower()):
        try:
            return np.array(mcolors.to_rgb(cand), np.float32)
        except ValueError:
            pass
    raise ValueError(f"无法解析颜色: {c}")


def _num(v):
    # 能全部转成 float 就返回 list[float]，否则返回 None（说明是类别名）
    try:
        return [float(a) for a in v]
    except (TypeError, ValueError):
        return None


def _txt(d, *path):
    for p in path:
        if not isinstance(d, dict):
            return ""
        d = d.get(p, {})
    return d if isinstance(d, str) else ""


def figureqa_to_spec(ann):
    t = ann.get("type")
    models = ann.get("models", [])
    info = ann.get("general_figure_info", {})
    spec = dict(title=_txt(info, "title", "text"),
                xlabel=_txt(info, "x_axis", "label", "text"),
                ylabel=_txt(info, "y_axis", "label", "text"))

    if t in ("line", "dot_line"):
        series = []
        for m in models:
            x, y = _num(m.get("x", [])), _num(m.get("y", []))
            if not x or not y or "color" not in m:
                continue
            series.append(dict(name=str(m.get("label", m.get("name", ""))), x=x, y=y, color=to_rgb01(m["color"])))
        spec.update(type=t, series=series[:K_MAX])

    elif t in ("vbar_categorical", "hbar_categorical"):
        series = []
        for m in models:
            cols = m.get("colors") or ([m["color"]] if "color" in m else [])
            xs, ys = m.get("x", []), m.get("y", [])
            if not isinstance(xs, list):            # 有的版本每根柱子单独一个 model
                xs, ys = [xs], [ys]
            nx, ny = _num(xs), _num(ys)
            if t.startswith("vbar"):                 # 竖柱：一般 x=类别, y=数值
                vals, labels = (ny, xs) if ny is not None else (nx, ys)
            else:                                    # 横柱：一般 x=数值, y=类别
                vals, labels = (nx, ys) if nx is not None else (ny, xs)
            labels = m.get("labels", labels)
            if vals is None:
                continue
            for v, lab, c in zip(vals, labels, cols):
                series.append(dict(name=str(lab), value=float(v), color=to_rgb01(c)))
        spec.update(type="vbar" if t.startswith("vbar") else "hbar", series=series[:K_MAX])
    else:
        return None                                   # 饼图等：暂不支持

    if len(spec["series"]) < 2:
        return None
    if RENAME_COLOR_LABELS:                           # 图例名字是颜色名，换色后会误导读者
        for i, s in enumerate(spec["series"]):
            s["name"] = f"Series {i + 1}" if spec["type"] in ("line", "dot_line") else f"C{i + 1}"
    return spec


def load_figureqa_specs(root, limit=0):
    path = os.path.join(root, "annotations.json")
    if not os.path.exists(path):
        print("没有 FigureQA annotations，跳过")
        return []
    anns = json.load(open(path))
    out, skipped = [], collections.Counter()
    for a in anns:
        s = figureqa_to_spec(a)
        if s is None:
            skipped[a.get("type")] += 1
            continue
        out.append((f"fqa{a.get('image_index', len(out)):06d}", s))
        if limit and len(out) >= limit:
            break
    print(f"FigureQA: 解析成功 {len(out)} 张 | 跳过: {dict(skipped)}")
    print("  类型分布:", collections.Counter(s["type"] for _, s in out))
    return out

## 5. Step 0'：随机生成合成图表 spec

作用：
1. **数据增广**：想要多少训练数据就生成多少
2. **补充 FigureQA 没有的图表类型**：散点图（你们的 proposal 里提到了，但 FigureQA 没有）
3. 没有 FigureQA 时整个 pipeline 也能跑

In [ ]:
def random_palette(k, rng, min_de=15.0, n_try=200):
    # 随机 HSV 配色，保证正常视觉下任意两个颜色 ΔE76 ≥ min_de（原图本身至少是"正常人能看清"的）
    best, best_s = None, -1
    for _ in range(n_try):
        hsv = np.stack([rng.random(k), rng.uniform(0.35, 1.0, k), rng.uniform(0.45, 0.95, k)], 1)
        c = mcolors.hsv_to_rgb(hsv).astype(np.float32)
        s = s_color(c)
        if s >= min_de:
            return c
        if s > best_s:
            best, best_s = c, s
    return best


def random_spec(rng):
    t = rng.choice(["line", "dot_line", "scatter", "vbar", "hbar"], p=[0.3, 0.15, 0.2, 0.2, 0.15])
    spec = dict(type=t, title=f"Chart {rng.integers(1000)}", xlabel="x", ylabel="y")
    if t in ("line", "dot_line"):
        k, n = int(rng.integers(2, 8)), int(rng.integers(8, 30))
        x = np.arange(n, dtype=float)
        series = [dict(name=f"Series {i + 1}", x=x.tolist(),
                       y=(rng.uniform(0, 50) + np.cumsum(rng.normal(0, 3, n))).tolist()) for i in range(k)]
    elif t == "scatter":
        k = int(rng.integers(2, 7))
        series = []
        for i in range(k):
            n = int(rng.integers(15, 40)); cx, cy = rng.uniform(0, 10, 2); sp = rng.uniform(0.5, 2.0)
            series.append(dict(name=f"Series {i + 1}", x=(cx + sp * rng.normal(size=n)).tolist(),
                               y=(cy + sp * rng.normal(size=n)).tolist()))
    else:
        k = int(rng.integers(3, K_MAX + 1))
        series = [dict(name=f"C{i + 1}", value=float(rng.uniform(5, 100))) for i in range(k)]
    for s, c in zip(series, random_palette(len(series), rng)):
        s["color"] = c
    spec["series"] = series
    return spec


_rng = np.random.default_rng(SEED)
print([random_spec(_rng)["type"] for _ in range(10)])

## 6. Step ①：决定输入图的配色

- 以 `1 - HARD_PALETTE_FRAC` 的概率：**保留原始配色**（FigureQA 原色 / 随机配色）
- 以 `HARD_PALETTE_FRAC` 的概率：**换成"难区分"配色**——随机采样很多组颜色，选出 *正常视觉下能区分（ΔE ≥ 15）但在某种 CVD 下最难区分* 的那一组（比如红配绿）

为什么需要第二种：FigureQA 原色很多本来就不难区分，target 和 input 几乎一样，模型学不到什么。

In [ ]:
def hard_palette(k, rng, cvd_idx, n_try=400, min_normal_de=15.0):
    # 一次性采样 n_try 组配色 (向量化)，选出: 正常视觉 min ΔE ≥ 15，且 CVD 模拟后 min ΔE 最小的一组
    hsv = np.stack([rng.random((n_try, k)), rng.uniform(0.4, 1.0, (n_try, k)), rng.uniform(0.45, 0.95, (n_try, k))], -1)
    pal = mcolors.hsv_to_rgb(hsv).astype(np.float32)                       # (n_try, k, 3)

    def min_pair(lab):
        d = np.linalg.norm(lab[:, :, None] - lab[:, None, :], axis=-1)    # (n_try, k, k)
        d[:, np.arange(k), np.arange(k)] = np.inf
        return d.min((1, 2))

    normal = min_pair(lab_np(pal))
    sim = min_pair(lab_np(simulate_cvd_np(pal, cvd_idx)))
    sim[normal < min_normal_de] = np.inf                                   # 原图正常人要能看清
    i = int(sim.argmin())
    if not np.isfinite(sim[i]):                                            # 极少数情况：都不满足 → 退回普通随机配色
        return random_palette(k, rng)
    return pal[i]


_rng = np.random.default_rng(0)
_p = hard_palette(4, _rng, CVD_INDEX["deuteranopia"])
print("正常视觉 S76 = %.1f | 绿色盲模拟 S76 = %.1f" % (s_color(_p), s_color(_p, CVD_INDEX["deuteranopia"])))

## 7. 渲染器：同一份 spec → RGB 图 / 像素对齐的系列 mask

关键设计：

1. **坐标轴位置固定**（`fig.add_axes([...])`，不用 `tight_layout`）→ 标题、图例、刻度标签不会影响绘图区域，所以 input / target / mask 三张图的数据点**像素级对齐**。
2. **mask 模式**：每个系列用唯一的"编号颜色" `(i+1, 0, 0)/255` 绘制，**关闭抗锯齿**，隐藏坐标轴和文字 → 解码后得到 `(H, W)` 的 uint8 图，0 = 背景/坐标轴/文字，`i+1` = 第 i 个系列。图例里的色块也会被标上对应编号（它们也需要被重新上色）。
3. **cues**（第 9 节决定）：`markers` / `linestyles`（折线、散点）、`hatches`（柱状图）。mask 会画出 marker 和虚线的形状，所以 **target 的 mask 要用 target 的 cues 重新渲染**。

In [ ]:
MARKERS    = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
LINESTYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2, 1, 2)), (0, (8, 2)), (0, (2, 2))]
HATCHES    = ["", "//", "..", "xx", "\\\\", "++", "oo", "--", "||", "**"]


def id_color(i):
    # mask 编码：R 通道 = 系列编号 + 1，G=B=0
    return ((i + 1) / 255.0, 0.0, 0.0)


def render_chart(spec, colors=None, cues=None, mode="rgb", size=IMG_SIZE):
    '''
    spec   : 第 4/5 节的 chart spec
    colors : (K,3) 每个系列的颜色；mode="mask" 时忽略
    cues   : dict，可含 markers / linestyles / hatches (长度 K 的 list)；None = 不加额外线索
    mode   : "rgb"  → 返回 float32 (H,W,3) in [0,1]
             "mask" → 返回 uint8 (H,W)，0 = 非系列像素，i+1 = 第 i 个系列
    '''
    cues = cues or {}
    is_mask = mode == "mask"
    K = len(spec["series"])
    sc = size / 256.0                       # 线宽 / 字号随分辨率缩放
    fs = 6.0 * sc
    rc = {"lines.antialiased": not is_mask, "patch.antialiased": not is_mask,
          "text.antialiased": not is_mask, "hatch.linewidth": 0.8 * sc, "font.size": fs}
    with matplotlib.rc_context(rc):
        fig = Figure(figsize=(size / 100, size / 100), dpi=100, facecolor="white")
        canvas = FigureCanvasAgg(fig)
        ax = fig.add_axes([0.15, 0.14, 0.80, 0.76])          # ← 固定绘图区域：保证像素对齐
        col = [id_color(i) if is_mask else tuple(np.clip(colors[i], 0, 1)) for i in range(K)]
        markers, lstyles, hatches = cues.get("markers"), cues.get("linestyles"), cues.get("hatches")
        t = spec["type"]

        if t in ("line", "dot_line"):
            for i, s in enumerate(spec["series"]):
                mk = markers[i] if markers else ("o" if t == "dot_line" else None)
                ls = lstyles[i] if lstyles else "-"
                ax.plot(s["x"], s["y"], color=col[i], lw=2.2 * sc, ls=ls, marker=mk, ms=4.0 * sc,
                        markevery=max(1, len(s["x"]) // 10), label=s["name"])
        elif t == "scatter":
            for i, s in enumerate(spec["series"]):
                mk = markers[i] if markers else "o"
                ax.scatter(s["x"], s["y"], color=[col[i]], marker=mk, s=(3.2 * sc) ** 2, linewidths=0,
                           antialiased=not is_mask, label=s["name"])
        elif t in ("vbar", "hbar"):
            pos = np.arange(K)
            vals = [s["value"] for s in spec["series"]]
            bars = ax.bar(pos, vals, color=col, width=0.7) if t == "vbar" else ax.barh(pos, vals, color=col, height=0.7)
            for i, p in enumerate(bars):
                p.set_linewidth(0)
                if hatches and hatches[i] and not is_mask:
                    p.set_hatch(hatches[i])
                    hc = tuple(np.clip(np.asarray(col[i]) * 0.45, 0, 1))   # 纹理用同色系的深色
                    if hasattr(p, "set_hatchcolor"):                       # matplotlib ≥ 3.10
                        p.set_hatchcolor(hc)
                    else:                                                  # 旧版本：纹理颜色 = 边框颜色
                        p.set_edgecolor(hc)
            if not is_mask:
                names = [s["name"] for s in spec["series"]]
                if t == "vbar":
                    ax.set_xticks(pos, names, rotation=45, ha="right", fontsize=fs * 0.85)
                else:
                    ax.set_yticks(pos, names, fontsize=fs * 0.85)
        else:
            raise ValueError(t)

        if t in ("line", "dot_line", "scatter"):              # 图例位置固定，rgb/mask 一致
            leg = ax.legend(loc="upper right", fontsize=fs * 0.8, frameon=not is_mask, handlelength=2.2)
            if is_mask:
                for txt in leg.get_texts():
                    txt.set_color("white")                    # 文字画成白色 = 不进 mask
        if is_mask:
            ax.axis("off")                                    # 坐标轴、刻度、标题都不进 mask
        else:
            ax.set_title(spec.get("title", ""), fontsize=fs)
            ax.set_xlabel(spec.get("xlabel", ""), fontsize=fs * 0.9)
            ax.set_ylabel(spec.get("ylabel", ""), fontsize=fs * 0.9)

        canvas.draw()
        arr = np.asarray(canvas.buffer_rgba())[..., :3].copy()

    if not is_mask:
        return arr.astype(np.float32) / 255.0
    r, g, b = (arr[..., i].astype(np.int32) for i in range(3))
    ids = np.where((g == 0) & (b == 0) & (r >= 1) & (r <= K), r, 0)
    return ids.astype(np.uint8)


# ---- 自检：mask 和 rgb 是否像素对齐 ----
_rng = np.random.default_rng(1)
for _t in ["line", "dot_line", "scatter", "vbar", "hbar"]:
    while True:
        _s = random_spec(_rng)
        if _s["type"] == _t:
            break
    _c = np.stack([s["color"] for s in _s["series"]])
    _img, _m = render_chart(_s, _c), render_chart(_s, mode="mask")
    # 在 mask 标为系列 i 的像素上，rgb 图应该就是系列 i 的颜色（允许抗锯齿边缘有少量偏差）
    err = [np.median(np.abs(_img[_m == i + 1] - _c[i]).max(-1)) for i in range(len(_c)) if (_m == i + 1).any()]
    print(f"{_t:9s} K={len(_c)} | mask 中出现的系列: {len(err)}/{len(_c)} | rgb-mask 颜色误差中位数: {max(err):.3f}")

## 8. Step ②：配色优化（生成 target 的颜色）

**这一步不训练任何网络。** 对每张图的输入配色，直接在颜色上做梯度下降，最小化你们原来的 3 项 loss：

$$c^*_{cvd} = \arg\min_{c} \; \lambda_1 L_{acc}(c, cvd) + \lambda_2 L_{struct}(c_{orig}, c) + \lambda_3 L_{vis}(c_{orig}, c)$$

- 4 种 CVD 各优化一次 → 4 组目标颜色
- 在 logit 空间优化（`sigmoid` 保证颜色始终在 [0,1]）
- **批量**：一次把 64 张图 × 4 种 CVD = 256 行一起优化（padding + mask，和旧 notebook 训练时的打包方式一样）

> 这也是你们旧 notebook 缺的 **"per-chart optimization" upper bound**：报告里可以说 *旧的调色板网络能达到这个最优解的 X%*。

In [ ]:
def optimize_palettes(orig_list, steps=OPT_STEPS, lr=OPT_LR, lam=LAM_TARGET, charts_per_batch=64, verbose=True):
    '''
    orig_list: list，每个元素是一张图的输入配色 (K_i, 3)
    返回: list，每个元素是 (4, K_i, 3) —— 4 种 CVD 各自的最优配色
    '''
    out, n_cvd = [], len(CVD_TYPES)
    t0 = time.time()
    for b0 in range(0, len(orig_list), charts_per_batch):
        chunk = orig_list[b0:b0 + charts_per_batch]
        B = len(chunk) * n_cvd
        old = torch.full((B, K_MAX, 3), 0.5)
        m = torch.zeros(B, K_MAX, dtype=torch.bool)
        for j, c in enumerate(chunk):                             # 每张图复制 4 行，每行对应一种 CVD
            k = min(len(c), K_MAX)
            old[j * n_cvd:(j + 1) * n_cvd, :k] = torch.tensor(np.asarray(c[:k]), dtype=torch.float32)
            m[j * n_cvd:(j + 1) * n_cvd, :k] = True
        old, m = old.to(DEVICE), m.to(DEVICE)
        cvd = torch.arange(n_cvd, device=DEVICE).repeat(len(chunk))
        bg = torch.ones(B, 3, device=DEVICE)                      # 渲染背景是白色

        z = torch.logit(old.clamp(1e-3, 1 - 1e-3)).clone().requires_grad_(True)   # 从原配色出发
        opt = torch.optim.Adam([z], lr=lr)
        for _ in range(steps):
            new = torch.sigmoid(z)
            loss, _ = total_loss(bg, old, new, m, cvd, lam=lam)
            opt.zero_grad(); loss.backward(); opt.step()

        res = torch.sigmoid(z).detach().cpu().numpy()
        for j, c in enumerate(chunk):
            out.append(res[j * n_cvd:(j + 1) * n_cvd, :min(len(c), K_MAX)])
        if verbose:
            print(f"  optimized {len(out)}/{len(orig_list)} charts  ({time.time() - t0:.0f}s)", flush=True)
    return out


# 小测试：一组红绿难区分的配色
_demo = hard_palette(4, np.random.default_rng(0), CVD_INDEX["deuteranopia"])
_opt = optimize_palettes([_demo], verbose=False)[0]
for v, name in enumerate(CVD_TYPES):
    print(f"{name:13s} S76: {s_color(_demo, v):5.1f} → {s_color(_opt[v], v):5.1f} | 颜色改变量 F = {palette_change(_demo, _opt[v]):.1f}")

## 9. Step ③：规则添加非颜色线索（marker / 线型 / 纹理）

只靠颜色有上限：比如全色盲 + 8 个系列，亮度轴上根本放不下 8 个都相差 30 ΔE 的颜色。所以：

| 条件 | 折线图 | 散点图 | 柱状图 |
|---|---|---|---|
| CVD = 全色盲 **或** 优化后 S76 < `CUE_THRESHOLD` | 不同 marker + 不同线型 | 不同 marker | 不同纹理 (hatch) |
| 其他 | 只改颜色 | 只改颜色 | 只改颜色 |

这样 target 里 **同时有"只改色"和"改色 + 加线索"两类样本**，模型要学会自己判断什么时候需要加线索——这正是你们 proposal 里的 *multi-cue redesign* 扩展。

In [ ]:
def decide_cues(chart_type, new_colors, cvd_idx, use_cues=USE_CUES):
    '返回 (cues dict, cue 类型名称)'
    K = len(new_colors)
    S = s_color(new_colors, cvd_idx)
    need = use_cues and (CVD_TYPES[cvd_idx] == "monochromacy" or S < CUE_THRESHOLD)
    if not need:
        return {}, "none"
    if chart_type in ("line", "dot_line"):
        return dict(markers=MARKERS[:K], linestyles=LINESTYLES[:K]), "markers+linestyles"
    if chart_type == "scatter":
        return dict(markers=MARKERS[:K]), "markers"
    return dict(hatches=HATCHES[:K]), "hatches"


for v in range(4):
    print(CVD_TYPES[v], "→", decide_cues("line", _opt[v], v)[1])

## 10. Step ④：生成并保存整个配对数据集

对每张图：
1. 选输入配色（第 6 节）→ 渲染 `input.png` + `input_mask.png`
2. 批量配色优化（第 8 节）→ 4 组目标颜色
3. 每种 CVD：决定 cues（第 9 节）→ 渲染 `{cvd}.png` + `{cvd}_mask.png`

**按"图"划分** train / val / test = 70 / 15 / 15（同一张图的 4 个 CVD 版本一定在同一个 split，避免泄漏）。

输出目录结构：
```
OUT_DIR/
  images/{cid}_in.png             输入图
  images/{cid}_in_mask.png        输入图的系列 mask
  images/{cid}_{cvd}.png          该 CVD 的 target 图
  images/{cid}_{cvd}_mask.png     target 的系列 mask（有 marker/虚线时和输入不同）
  meta.jsonl                      每行一张图：split、类型、输入颜色、每种 CVD 的目标颜色/cues/分数
```

⏱ 大概速度：渲染 ≈ 0.2–0.4 秒/张图（共 10 张小图），2000 张图约 10 分钟。

In [ ]:
def build_dataset(out_dir=OUT_DIR, n_figureqa=N_FIGUREQA, n_synth=N_SYNTH, seed=SEED):
    rng = np.random.default_rng(seed)
    os.makedirs(os.path.join(out_dir, "images"), exist_ok=True)

    # ---- Step 0: 收集 chart specs ----
    charts = [dict(cid=cid, source="figureqa", spec=s) for cid, s in load_figureqa_specs(DATA_ROOT, n_figureqa)]
    charts += [dict(cid=f"syn{i:06d}", source="synthetic", spec=random_spec(rng)) for i in range(n_synth)]
    print(f"总共 {len(charts)} 张图")

    # ---- Step ①: 输入配色 ----
    for ch in charts:
        K = len(ch["spec"]["series"])
        if rng.random() < HARD_PALETTE_FRAC:
            ch["hard_for"] = CVD_TYPES[int(rng.integers(0, 3))]           # 针对哪种（有颜色的）CVD 选的难配色
            ch["in_colors"] = hard_palette(K, rng, CVD_INDEX[ch["hard_for"]])
        else:
            ch["hard_for"] = None
            ch["in_colors"] = np.stack([s["color"] for s in ch["spec"]["series"]]).astype(np.float32)

    # ---- Step ②: 批量配色优化 ----
    print("Step ②: 配色优化 ...")
    targets = optimize_palettes([ch["in_colors"] for ch in charts])

    # ---- 按图划分 split ----
    perm = rng.permutation(len(charts))
    n_tr, n_va = int(0.7 * len(charts)), int(0.15 * len(charts))
    split_of = {}
    for r, i in enumerate(perm):
        split_of[i] = "train" if r < n_tr else ("val" if r < n_tr + n_va else "test")

    # ---- Step ③④: 决定 cues、渲染、保存 ----
    print("Step ③④: 渲染 ...")
    img_dir, t0 = os.path.join(out_dir, "images"), time.time()
    save_rgb = lambda a, p: Image.fromarray((np.clip(a, 0, 1) * 255).round().astype(np.uint8)).save(p)
    with open(os.path.join(out_dir, "meta.jsonl"), "w") as fh:
        for i, (ch, tgt) in enumerate(zip(charts, targets)):
            spec, cid, cin = ch["spec"], ch["cid"], ch["in_colors"]
            m_in = render_chart(spec, mode="mask")
            save_rgb(render_chart(spec, cin), f"{img_dir}/{cid}_in.png")
            Image.fromarray(m_in).save(f"{img_dir}/{cid}_in_mask.png")
            rec = dict(cid=cid, source=ch["source"], split=split_of[i], type=spec["type"], K=len(cin),
                       hard_for=ch["hard_for"], in_colors=np.round(cin, 4).tolist(), targets={})
            for v, cvd_name in enumerate(CVD_TYPES):
                cues, cue_name = decide_cues(spec["type"], tgt[v], v)
                save_rgb(render_chart(spec, tgt[v], cues), f"{img_dir}/{cid}_{cvd_name}.png")
                m_t = render_chart(spec, cues=cues, mode="mask") if cues else m_in   # 没加线索时 mask 和输入相同
                Image.fromarray(m_t).save(f"{img_dir}/{cid}_{cvd_name}_mask.png")
                rec["targets"][cvd_name] = dict(colors=np.round(tgt[v], 4).tolist(), cues=cue_name,
                                                S76_in=s_color(cin, v), S76_tgt=s_color(tgt[v], v),
                                                F=palette_change(cin, tgt[v]))
            fh.write(json.dumps(rec) + "\n")
            if (i + 1) % 200 == 0 or i + 1 == len(charts):
                el = time.time() - t0
                print(f"  {i + 1}/{len(charts)}  ({el:.0f}s, 预计还需 {el / (i + 1) * (len(charts) - i - 1):.0f}s)", flush=True)
    print("完成 →", out_dir)


def load_meta(out_dir=OUT_DIR):
    return [json.loads(l) for l in open(os.path.join(out_dir, "meta.jsonl"))]


build_dataset()
META = load_meta()
print(collections.Counter(m["split"] for m in META), collections.Counter(m["type"] for m in META))

## 11. 检查：可视化几组样本

每行：输入 | 输入在 CVD 下的样子 | target | target 在 CVD 下的样子 | target mask（彩色显示）

**重点看**：target 在 CVD 模拟下是否明显比输入好区分；mask 是否和图中的线/柱子对齐。

In [ ]:
def load_png(p):
    return np.asarray(Image.open(p).convert("RGB"), np.float32) / 255.0


def colorize_mask(m):
    pal = np.vstack([[1, 1, 1], plt.cm.tab10(np.arange(10))[:, :3]])
    return pal[np.clip(m, 0, 10)]


def show_pairs(meta, n=4, cvds=("deuteranopia", "monochromacy"), seed=0):
    rng = np.random.default_rng(seed)
    picks = [meta[i] for i in rng.choice(len(meta), n, replace=False)]
    rows = [(m, c) for m in picks for c in cvds]
    fig, axes = plt.subplots(len(rows), 5, figsize=(15, 3 * len(rows)))
    img_dir = os.path.join(OUT_DIR, "images")
    for r, (m, c) in enumerate(rows):
        v = CVD_INDEX[c]
        x = load_png(f"{img_dir}/{m['cid']}_in.png")
        y = load_png(f"{img_dir}/{m['cid']}_{c}.png")
        mt = np.asarray(Image.open(f"{img_dir}/{m['cid']}_{c}_mask.png"))
        tinfo = m["targets"][c]
        panels = [(x, "input"), (simulate_cvd_np(x, v), f"input @ {c}\nS76={tinfo['S76_in']:.1f}"),
                  (y, f"target ({tinfo['cues']})"), (simulate_cvd_np(y, v), f"target @ {c}\nS76={tinfo['S76_tgt']:.1f}"),
                  (colorize_mask(mt), "target mask")]
        for a, (im, title) in zip(axes[r], panels):
            a.imshow(im); a.set_title(title, fontsize=9); a.axis("off")
    plt.tight_layout(); plt.show()


show_pairs(META)

## 12. 检查：target 质量统计

每种 CVD：输入的 S76 → target 的 S76（越高越好），颜色改变量 F（越低越接近原图），加了线索的比例。

**怎么用这张表调 `LAM_TARGET`：**
- ΔS 很小（< 5）→ target 和输入几乎一样，模型学不到东西 → 调小 λ₃（visual）或 λ₂
- F 很大（> 40）且图看起来完全不像原图 → 调大 λ₃
- 改完要重新运行第 10 节

In [ ]:
def target_stats(meta, split=None):
    rows = [m for m in meta if split is None or m["split"] == split]
    print(f"{len(rows)} charts" + (f" ({split})" if split else ""))
    print(f"{'cvd':14s}{'S76 in':>9s}{'S76 tgt':>9s}{'ΔS76':>8s}{'F':>8s}{'cues%':>8s}")
    for c in CVD_TYPES:
        si = np.array([m["targets"][c]["S76_in"] for m in rows])
        st = np.array([m["targets"][c]["S76_tgt"] for m in rows])
        f = np.array([m["targets"][c]["F"] for m in rows])
        cu = np.mean([m["targets"][c]["cues"] != "none" for m in rows]) * 100
        print(f"{c:14s}{np.nanmean(si):9.1f}{np.nanmean(st):9.1f}{np.nanmean(st - si):8.1f}{np.mean(f):8.1f}{cu:7.0f}%")
    print("\n按图表类型的 cue 使用:", collections.Counter((m["type"], m["targets"]["deuteranopia"]["cues"]) for m in rows))


target_stats(META)

## 13. PyTorch Dataset

每个样本 = (一张图, 一种 CVD)。所以数据集大小 = 图数 × 4。

返回：`x` 输入图 (3,H,W)、`y` target 图 (3,H,W)、`mi` 输入 mask (H,W)、`mt` target mask (H,W)、`c` CVD 编号。

In [ ]:
class CVDPairDataset(Dataset):
    def __init__(self, out_dir, split, cvds=tuple(range(len(CVD_TYPES)))):
        self.dir = os.path.join(out_dir, "images")
        self.items = [(m["cid"], c) for m in load_meta(out_dir) if m["split"] == split for c in cvds]

    def __len__(self):
        return len(self.items)

    def _img(self, name):
        a = np.asarray(Image.open(os.path.join(self.dir, name)).convert("RGB"), np.float32) / 255.0
        return torch.from_numpy(a).permute(2, 0, 1)

    def _mask(self, name):
        return torch.from_numpy(np.asarray(Image.open(os.path.join(self.dir, name)), np.int64))

    def __getitem__(self, i):
        cid, c = self.items[i]
        cn = CVD_TYPES[c]
        return (self._img(f"{cid}_in.png"), self._img(f"{cid}_{cn}.png"),
                self._mask(f"{cid}_in_mask.png"), self._mask(f"{cid}_{cn}_mask.png"), c)


_ds = CVDPairDataset(OUT_DIR, "train")
_x, _y, _mi, _mt, _c = _ds[0]
print(len(_ds), "train samples |", _x.shape, _y.shape, _mi.shape, _mt.shape, CVD_TYPES[_c])

## 14. 模型：CVD 条件 U-Net（FiLM conditioning）

- 标准 U-Net：4 层编码器（256→128→64→32）+ 解码器 + skip connections
- **CVD 条件**：CVD 编号 → embedding → 每个残差块里用 FiLM（$h \leftarrow h \cdot (1+\gamma) + \beta$）调制特征。这是 CVD-conditioned generation 的标准做法
- **残差输出 + 恒等初始化**：输出 = `sigmoid(logit(x) + Δ)`，Δ 的卷积层初始化为 0 → 训练一开始模型就是"什么都不改"（和旧 PaletteRecolorNet 同样的思路）
- `base=32` 约 3.4M 参数，Colab T4 上 256×256 batch 16 没问题

In [ ]:
class FiLMResBlock(nn.Module):
    def __init__(self, cin, cout, cdim):
        super().__init__()
        self.c1, self.n1 = nn.Conv2d(cin, cout, 3, padding=1), nn.GroupNorm(8, cout)
        self.c2, self.n2 = nn.Conv2d(cout, cout, 3, padding=1), nn.GroupNorm(8, cout)
        self.film = nn.Linear(cdim, 2 * cout)                      # CVD embedding → (γ, β)
        nn.init.zeros_(self.film.weight); nn.init.zeros_(self.film.bias)
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x, e):
        h = F.silu(self.n1(self.c1(x)))
        g, b = self.film(e).chunk(2, -1)
        h = h * (1 + g[..., None, None]) + b[..., None, None]      # FiLM 调制
        h = F.silu(self.n2(self.c2(h)))
        return h + self.skip(x)


class CVDUNet(nn.Module):
    def __init__(self, base=32, n_cvd=len(CVD_TYPES), cdim=64):
        super().__init__()
        self.emb = nn.Sequential(nn.Embedding(n_cvd, cdim), nn.Linear(cdim, cdim), nn.SiLU())
        ch = [base, base * 2, base * 4, base * 8]
        self.inc = nn.Conv2d(3, base, 3, padding=1)
        self.enc = nn.ModuleList([FiLMResBlock(base if i == 0 else ch[i - 1], ch[i], cdim) for i in range(4)])
        self.mid = FiLMResBlock(ch[3], ch[3], cdim)
        self.dec = nn.ModuleList([FiLMResBlock(ch[i + 1] + ch[i], ch[i], cdim) for i in reversed(range(3))])
        self.out = nn.Conv2d(base, 3, 1)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)    # 恒等初始化

    def forward(self, x, cvd):
        e = self.emb(cvd)
        h = self.inc(x)
        skips = []
        for i, blk in enumerate(self.enc):
            h = blk(h, e)
            if i < 3:
                skips.append(h)
                h = F.max_pool2d(h, 2)
        h = self.mid(h, e)
        for blk in self.dec:
            h = F.interpolate(h, scale_factor=2, mode="bilinear", align_corners=False)
            h = blk(torch.cat([h, skips.pop()], 1), e)
        return torch.sigmoid(torch.logit(x.clamp(1e-3, 1 - 1e-3)) + self.out(h))


_net = CVDUNet()
print(f"{sum(p.numel() for p in _net.parameters()) / 1e6:.2f}M params")
with torch.no_grad():
    _o = _net(_x[None], torch.tensor([_c]))
print("output", tuple(_o.shape), "| 初始化时和输入的最大差:", float((_o - _x[None]).abs().max()))

## 15. 图像级 loss

$$L = w_{rec} \cdot \underbrace{\|\hat y - y\|_1}_{\text{模仿 target}} \;+\; w_{acc} \cdot \underbrace{L_{acc}(\text{series\_means}(\hat y, m_t), cvd)}_{\text{你们的无障碍 loss，作用在像素上}} \;+\; w_{keep} \cdot \underbrace{\|(\hat y - x) \odot \text{keep}\|_1}_{\text{坐标轴/文字/背景不许动}}$$

- **series_means**：用 target mask 把输出图里每个系列的像素取平均 → 得到 (K,3) 的颜色 → 喂给旧的 `accessibility_loss`。这让模型即使在 target 不完美时也直接被"可区分性"监督
- **keep 区域** = 输入 mask 和 target mask 都是 0 的像素（再膨胀 1 像素，排除抗锯齿边缘）→ 坐标轴、刻度、文字、背景必须原样保留。这就是 *structure preservation*，而且是真正的空间结构，不只是颜色

In [ ]:
def series_means(img, m, k_max=K_MAX, min_px=4):
    '''img (B,3,H,W), m (B,H,W) long → 每个系列的平均颜色 (B,K,3) 和有效标记 (B,K)'''
    oh = F.one_hot(m.clamp(max=k_max), k_max + 1).permute(0, 3, 1, 2).float()   # (B,K+1,H,W)
    cnt = oh.sum((2, 3))                                                       # (B,K+1)
    s = torch.einsum("bkhw,bchw->bkc", oh, img)
    mean = s / cnt.clamp(min=1)[..., None]
    return mean[:, 1:], (cnt > min_px)[:, 1:]                                   # 去掉 id=0 (背景)


def keep_region(mi, mt):
    series = ((mi > 0) | (mt > 0)).float()[:, None]
    series = F.max_pool2d(series, 3, stride=1, padding=1)                      # 膨胀 1 像素
    return 1.0 - series                                                        # (B,1,H,W)


def image_loss(out, x, y, mi, mt, cvd, w_rec=1.0, w_acc=0.1, w_keep=1.0):
    l_rec = (out - y).abs().mean()
    means, valid = series_means(out, mt)
    bg = torch.ones(out.shape[0], 3, device=out.device)
    l_acc = accessibility_loss(bg, means, valid, cvd) if w_acc > 0 else out.new_zeros(())
    keep = keep_region(mi, mt)
    l_keep = ((out - x).abs() * keep).sum() / (keep.sum() * 3).clamp(min=1)
    loss = w_rec * l_rec + w_acc * l_acc + w_keep * l_keep
    return loss, dict(rec=l_rec.item(), acc=float(l_acc), keep=l_keep.item())


with torch.no_grad():
    print(image_loss(_x[None], _x[None], _y[None], _mi[None], _mt[None], torch.tensor([_c]))[1])

## 16. 训练

- AdamW + cosine 学习率
- **验证集是固定的**（每张 val 图 × 全部 4 种 CVD，不随机替换样本）→ 修复了旧 notebook 报告里提到的"验证 loss 是随机的"问题
- **按验证 loss 保存最好的 checkpoint**（旧 notebook 用的是最后一个 epoch）
- 消融实验通过改 `cfg` 实现（第 20 节）

In [ ]:
BASE_TRAIN_CFG = dict(epochs=EPOCHS, bs=BATCH_SIZE, lr=2e-4, base=32,
                      w_rec=1.0, w_acc=0.1, w_keep=1.0,
                      no_cond=False,              # True = 消融：模型看不到 CVD 类型
                      out="unet_full.pt", seed=SEED)


def run_epoch(net, dl, cfg, opt=None, sched=None):
    train = opt is not None
    net.train(train)
    agg, n = collections.defaultdict(float), 0
    with torch.set_grad_enabled(train):
        for x, y, mi, mt, c in dl:
            x, y, mi, mt, c = x.to(DEVICE), y.to(DEVICE), mi.to(DEVICE), mt.to(DEVICE), c.to(DEVICE)
            cin = torch.zeros_like(c) if cfg["no_cond"] else c        # 消融：给模型一个常数条件
            out = net(x, cin)
            loss, parts = image_loss(out, x, y, mi, mt, c, cfg["w_rec"], cfg["w_acc"], cfg["w_keep"])
            if train:
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                opt.step(); sched.step()
            agg["loss"] += loss.item() * len(x)
            for k, v in parts.items():
                agg[k] += v * len(x)
            n += len(x)
    return {k: v / n for k, v in agg.items()}


def train_unet(**overrides):
    cfg = {**BASE_TRAIN_CFG, **overrides}
    torch.manual_seed(cfg["seed"])
    dl_tr = DataLoader(CVDPairDataset(OUT_DIR, "train"), batch_size=cfg["bs"], shuffle=True,
                       num_workers=2, drop_last=True, pin_memory=True)
    dl_va = DataLoader(CVDPairDataset(OUT_DIR, "val"), batch_size=cfg["bs"], shuffle=False, num_workers=2)
    net = CVDUNet(base=cfg["base"]).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=cfg["lr"], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"] * len(dl_tr))
    best, hist = float("inf"), []
    for ep in range(cfg["epochs"]):
        t0 = time.time()
        tr = run_epoch(net, dl_tr, cfg, opt, sched)
        va = run_epoch(net, dl_va, cfg)
        hist.append(dict(epoch=ep, **{f"train_{k}": v for k, v in tr.items()}, **{f"val_{k}": v for k, v in va.items()}))
        flag = ""
        if va["loss"] < best:
            best = va["loss"]; flag = " *saved*"
            torch.save(dict(model=net.state_dict(), cfg=cfg, epoch=ep), cfg["out"])
        print(f"ep {ep:3d} | train {tr['loss']:.4f} (rec {tr['rec']:.4f} acc {tr['acc']:.3f} keep {tr['keep']:.4f}) "
              f"| val {va['loss']:.4f} (rec {va['rec']:.4f} acc {va['acc']:.3f} keep {va['keep']:.4f}) | {time.time() - t0:.0f}s{flag}")
    json.dump(hist, open(cfg["out"] + ".hist.json", "w"))
    return hist


def load_unet(ckpt):
    ck = torch.load(ckpt, map_location=DEVICE)
    net = CVDUNet(base=ck["cfg"]["base"]).to(DEVICE)
    net.load_state_dict(ck["model"]); net.eval()
    return net, ck["cfg"]


hist_full = train_unet()

In [ ]:
# 训练曲线
fig, ax = plt.subplots(1, 4, figsize=(16, 3))
for a, k in zip(ax, ["loss", "rec", "acc", "keep"]):
    a.plot([h[f"train_{k}"] for h in hist_full], label="train")
    a.plot([h[f"val_{k}"] for h in hist_full], label="val")
    a.set_title(k); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 17. 测试集评估 + 固定调色板 baseline

对每个测试样本，**从像素里**测量每个系列的颜色（用输入 mask 和 target mask 都标同一系列的"核心像素"），然后计算：

| 指标 | 含义 |
|---|---|
| `S00_in / S00_out / S00_tgt` | CVD 模拟下最难区分那对颜色的 CIEDE2000（**训练没优化过 CIEDE2000**，所以比 CIE76 更公平） |
| `dS00` | 输出相对输入的提升 |
| `F` | 正常视觉下颜色改变量（越小越像原图） |
| `L1_tgt` | 和 target 的像素 L1 |
| `PSNR_keep` | 非系列区域（坐标轴/文字/背景）的 PSNR，越高说明结构保持越好 |

**Baselines**（只改颜色，按调色板算）：Okabe–Ito、Petroff-10。这次用 **Hungarian 匹配**（每个原色配给最接近的调色板颜色，最小化 F），比旧 notebook 按顺序分配更公平。

In [ ]:
def _hex(lst):
    return np.array([[int(h[i:i + 2], 16) for i in (0, 2, 4)] for h in lst], np.float32) / 255.0


FIXED_PALETTES = {   # 和旧 notebook 一致（Petroff 的 hex 值记得对照 arXiv:2107.02270 核实）
    "okabe_ito": _hex(["E69F00", "56B4E9", "009E73", "F0E442", "0072B2", "D55E00", "CC79A7", "000000"]),
    "petroff10": _hex(["3f90da", "ffa90e", "bd1f01", "94a4a2", "832db6", "a96b59", "e76300", "b9ac70", "717581", "92dadd"]),
}


def fixed_assign(old, pal):
    # Hungarian 匹配：每个原色分配一个不同的调色板颜色，使总 ΔE 最小
    K = len(old)
    if K > len(pal):
        pal = np.concatenate([pal] * math.ceil(K / len(pal)))
    cost = np.linalg.norm(lab_np(old)[:, None] - lab_np(pal)[None], axis=-1)
    r, c = linear_sum_assignment(cost)
    new = np.zeros_like(old); new[r] = pal[c]
    return new


@torch.no_grad()
def evaluate_unet(ckpt, split="test", max_k=8):
    net, cfg = load_unet(ckpt)
    ds = CVDPairDataset(OUT_DIR, split)
    dl = DataLoader(ds, batch_size=32, shuffle=False, num_workers=2)
    rows = []
    for x, y, mi, mt, c in dl:
        xd, cd = x.to(DEVICE), c.to(DEVICE)
        out = net(xd, torch.zeros_like(cd) if cfg["no_cond"] else cd).cpu()
        core = torch.where((mi == mt) & (mi > 0), mi, torch.zeros_like(mi))     # 两个 mask 都同意的系列像素
        m_in, val = series_means(x, core)
        m_out, _ = series_means(out, core)
        m_tg, _ = series_means(y, core)
        for b in range(len(x)):
            v = int(c[b]); idx = val[b].nonzero().flatten()
            if len(idx) < 2 or len(idx) > max_k:
                continue
            ci, co, ct = (t[b, idx].numpy() for t in (m_in, m_out, m_tg))
            keep = keep_region(mi[b:b + 1], mt[b:b + 1])[0, 0].bool()
            mse = float(((out[b] - x[b]) ** 2)[:, keep].mean())
            r = dict(cvd=CVD_TYPES[v], S00_in=s_color(ci, v, "ciede2000"), S00_out=s_color(co, v, "ciede2000"),
                     S00_tgt=s_color(ct, v, "ciede2000"), F=palette_change(ci, co),
                     L1_tgt=float((out[b] - y[b]).abs().mean()), PSNR_keep=10 * math.log10(1 / max(mse, 1e-10)))
            for name, pal in FIXED_PALETTES.items():
                fx = fixed_assign(ci, pal)
                r[f"S00_{name}"], r[f"F_{name}"] = s_color(fx, v, "ciede2000"), palette_change(ci, fx)
            rows.append(r)
    return rows


def summarize(rows, title=""):
    print(f"\n=== {title}  ({len(rows)} test samples, ≤8 series) ===")
    hdr = ["dS00 ours", "F ours", "dS00 tgt", "dS00 okabe", "F okabe", "dS00 petroff", "F petroff", "L1_tgt", "PSNR_keep"]
    print(f"{'cvd':14s}" + "".join(f"{h:>13s}" for h in hdr))
    for c in CVD_TYPES:
        R = [r for r in rows if r["cvd"] == c]
        g = lambda k: np.nanmean([r[k] for r in R])
        vals = [g("S00_out") - g("S00_in"), g("F"), g("S00_tgt") - g("S00_in"),
                g("S00_okabe_ito") - g("S00_in"), g("F_okabe_ito"),
                g("S00_petroff10") - g("S00_in"), g("F_petroff10"), g("L1_tgt"), g("PSNR_keep")]
        print(f"{c:14s}" + "".join(f"{v:13.2f}" for v in vals))


res_full = evaluate_unet("unet_full.pt")
summarize(res_full, "U-Net full")

## 18. 定性对比图

每行：输入 | 输入@CVD | **模型输出** | 输出@CVD | target | target@CVD。可以直接放进报告。

In [ ]:
@torch.no_grad()
def show_predictions(ckpt, split="test", n=3, cvds=("deuteranopia", "tritanopia", "monochromacy"), seed=1, save_as=None):
    net, cfg = load_unet(ckpt)
    meta = [m for m in load_meta() if m["split"] == split]
    picks = [meta[i] for i in np.random.default_rng(seed).choice(len(meta), n, replace=False)]
    img_dir = os.path.join(OUT_DIR, "images")
    rows = [(m, c) for m in picks for c in cvds]
    fig, axes = plt.subplots(len(rows), 6, figsize=(18, 3 * len(rows)))
    for r, (m, c) in enumerate(rows):
        v = CVD_INDEX[c]
        x = load_png(f"{img_dir}/{m['cid']}_in.png")
        y = load_png(f"{img_dir}/{m['cid']}_{c}.png")
        xt = torch.from_numpy(x).permute(2, 0, 1)[None].to(DEVICE)
        o = net(xt, torch.tensor([0 if cfg["no_cond"] else v], device=DEVICE))[0].permute(1, 2, 0).cpu().numpy()
        panels = [(x, "input"), (simulate_cvd_np(x, v), f"input @ {c}"), (o, "ours"),
                  (simulate_cvd_np(o, v), f"ours @ {c}"), (y, "target"), (simulate_cvd_np(y, v), f"target @ {c}")]
        for a, (im, t) in zip(axes[r], panels):
            a.imshow(np.clip(im, 0, 1)); a.set_title(t, fontsize=9); a.axis("off")
    plt.tight_layout()
    if save_as:
        plt.savefig(save_as, dpi=150, bbox_inches="tight")
    plt.show()


show_predictions("unet_full.pt", save_as="qualitative_test.png")

## 19. 在真实图表（ChartQA）上推理：泛化测试

模型只在我们渲染的图上训练过。这一节测试 **合成 → 真实** 的泛化（报告里很有价值的一节，不管结果好坏都能写）。

1. 把几十张 ChartQA 测试集 PNG 放到 `REAL_DIR`
2. 图片先补白边成正方形，再缩放到 `IMG_SIZE`
3. 真实图没有 mask 也没有 target → 用旧 notebook 的 `extract_palette` 分别从输入和输出里提取颜色，比较 S00

下面先贴入旧 notebook 第 3 节的提取 / 重着色函数（原封不动）。

In [ ]:
def load_rgb(path):
    """Load PNG as float RGB in [0,1]; RGBA images (e.g. FigureQA) are composited over white."""
    im = Image.open(path).convert("RGBA")
    bg = Image.new("RGBA", im.size, (255, 255, 255, 255))
    return np.asarray(Image.alpha_composite(bg, im).convert("RGB"), dtype=np.float32) / 255.0


def _lab(x):
    return rgb2lab(x.reshape(-1, 1, 3)).reshape(-1, 3)


def interior_count(img, color, tol=0.03):
    """# of pixels within `tol` of `color` whose 4 neighbours are too: flat regions (bars, pie slices) have
    many; anti-aliasing blends along edges / thin lines have ~none."""
    m = np.linalg.norm(img - color, axis=-1) < tol
    inner = m.copy()
    inner[1:] &= m[:-1]; inner[:-1] &= m[1:]; inner[:, 1:] &= m[:, :-1]; inner[:, :-1] &= m[:, 1:]
    inner[0] = inner[-1] = False; inner[:, 0] = inner[:, -1] = False
    return int(inner.sum())


def find_background(img):
    q = (img.reshape(-1, 3) * 31).round().astype(np.int32)
    keys = q[:, 0] * 1024 + q[:, 1] * 32 + q[:, 2]
    vals, counts = np.unique(keys, return_counts=True)
    k = vals[counts.argmax()]
    mask = keys == k
    return img.reshape(-1, 3)[mask].mean(0)


def extract_palette(img, k_max=10, min_frac=0.0003, merge_de=12.0,
                    bg_tol=0.06, interior_min=8, ignore_neutral=True, max_pixels=200_000, seed=0):
    """Return dict(bg, colors(K,3), counts(K,)). Colors sorted by pixel count.
    min_frac is relative to ALL pixels (not just non-background), so small bars / thin lines survive.
    ignore_neutral drops every low-chroma color (text, axes, gridlines, black/gray series)."""
    rng = np.random.default_rng(seed)
    bg = find_background(img)
    px = img.reshape(-1, 3)
    if len(px) > max_pixels:
        px = px[rng.choice(len(px), max_pixels, replace=False)]
    n_total = len(px)
    px = px[np.linalg.norm(px - bg, axis=1) > bg_tol]          # drop background
    if ignore_neutral and len(px):                              # drop text/axes/gridlines
        lab = _lab(px)
        chroma = np.hypot(lab[:, 1], lab[:, 2])
        neutral = chroma < 8
        px = px[~neutral]
    if len(px) == 0:
        return dict(bg=bg, colors=np.zeros((0, 3), np.float32), counts=np.zeros(0))
    q = (px * 15).round().astype(np.int32)                      # 16 levels / channel
    keys = q[:, 0] * 256 + q[:, 1] * 16 + q[:, 2]
    vals, inv, counts = np.unique(keys, return_inverse=True, return_counts=True)
    order = np.argsort(-counts)
    cents = np.stack([px[inv == i].mean(0) for i in order])     # candidate centers
    cnts = counts[order]
    cl = _lab(cents)
    kept, assign = [], np.full(len(cents), -1)
    for i in range(len(cents)):                                 # greedy Lab merge
        if kept:
            d = np.linalg.norm(cl[kept] - cl[i], axis=1)
            j = int(d.argmin())
            if d[j] < merge_de:
                assign[i] = j
                continue
        assign[i] = len(kept)
        kept.append(i)
    K = len(kept)
    tot = np.zeros(K)
    sums = np.zeros((K, 3))
    for i, a in enumerate(assign):
        tot[a] += cnts[i]
        sums[a] += cents[i] * cnts[i]
    colors = sums / tot[:, None]
    keep = tot / n_total >= min_frac
    colors, tot = colors[keep], tot[keep]
    o = np.argsort(-tot)
    colors, tot = colors[o], tot[o]
    # drop anti-aliasing blends (colors without a flat interior):
    #  (1) on the segment between the background and a stronger color (edges of any shape / thin lines);
    #  (2) on the segment between two other series colors that are both >=5x more common
    #      (boundary pixels between adjacent pie slices / stacked regions).
    keep = np.ones(len(colors), bool)
    for i in range(len(colors)):
        is_blend = False
        for j in range(len(colors)):
            if j == i:
                continue
            d = colors[j] - bg
            dd = float(d @ d)
            if dd > 1e-6:
                t = float((colors[i] - bg) @ d) / dd
                if 0.02 < t < 0.95 and np.linalg.norm(colors[i] - (bg + t * d)) < 0.06:
                    is_blend = True
                    break
            for k in range(j + 1, len(colors)):
                if k == i or tot[j] < 5 * tot[i] or tot[k] < 5 * tot[i]:
                    continue
                d = colors[j] - colors[k]
                dd = float(d @ d)
                if dd < 1e-6:
                    continue
                al = float((colors[i] - colors[k]) @ d) / dd
                if 0.02 < al < 0.98 and np.linalg.norm(colors[i] - (colors[k] + al * d)) < 0.03:
                    is_blend = True
                    break
            if is_blend:
                break
        if is_blend and interior_count(img, colors[i]) < interior_min:
            keep[i] = False
    colors, tot = colors[keep][:k_max], tot[keep][:k_max]
    o = np.arange(len(colors))
    return dict(bg=bg.astype(np.float32), colors=colors[o].astype(np.float32), counts=tot[o])


def recolor_image(img, bg, old, new, res_tol=0.12, t_min=0.05, cos_min=0.97):
    """Apply old[k] -> new[k]. Pixels lying on the segment bg->old[k] (incl. anti-aliasing)
    are shifted by t * (new[k]-old[k]); everything else (text, axes) is untouched."""
    H, W, _ = img.shape
    px = img.reshape(-1, 3)
    out = px.copy()
    best_res = np.full(len(px), np.inf)
    best_k = np.full(len(px), -1)
    best_t = np.zeros(len(px))
    for k in range(len(old)):
        d = old[k] - bg
        dd = float(d @ d)
        if dd < 1e-6:
            continue
        t = np.clip(((px - bg) @ d) / dd, 0, 1)
        res = np.linalg.norm(px - (bg + t[:, None] * d), axis=1)
        v = px - bg
        cos = (v @ d) / (np.linalg.norm(v, axis=1) * np.linalg.norm(d) + 1e-8)
        ok = (res < res_tol) & (t > t_min) & (cos > cos_min) & (res < best_res)
        best_res[ok], best_k[ok], best_t[ok] = res[ok], k, t[ok]
    for k in range(len(old)):
        m = best_k == k
        if m.any():
            out[m] = px[m] + best_t[m, None] * (new[k] - old[k])
    return np.clip(out, 0, 1).reshape(H, W, 3)

In [ ]:
def prep_real(path, size=IMG_SIZE):
    # 补白边成正方形 → 缩放；返回 float32 (H,W,3)
    img = Image.fromarray((load_rgb(path) * 255).astype(np.uint8))
    s = max(img.size)
    canvas = Image.new("RGB", (s, s), (255, 255, 255))
    canvas.paste(img, ((s - img.size[0]) // 2, (s - img.size[1]) // 2))
    return np.asarray(canvas.resize((size, size), Image.LANCZOS), np.float32) / 255.0


@torch.no_grad()
def run_real(ckpt, real_dir=REAL_DIR, n_show=3, cvd="deuteranopia"):
    net, cfg = load_unet(ckpt)
    paths = sorted(glob.glob(os.path.join(real_dir, "*.png")))
    if not paths:
        print("REAL_DIR 里没有 PNG：放几张 ChartQA 图进去再运行"); return
    v = CVD_INDEX[cvd]
    stats = collections.defaultdict(list)
    shown = 0
    for p in paths:
        x = prep_real(p)
        xt = torch.from_numpy(x).permute(2, 0, 1)[None].to(DEVICE)
        for vv, cn in enumerate(CVD_TYPES):
            o = net(xt, torch.tensor([0 if cfg["no_cond"] else vv], device=DEVICE))[0].permute(1, 2, 0).cpu().numpy()
            pi, po = extract_palette(x)["colors"], extract_palette(o)["colors"]
            if len(pi) >= 2 and len(po) >= 2:
                stats[cn].append(s_color(po, vv, "ciede2000") - s_color(pi, vv, "ciede2000"))
            if cn == cvd and shown < n_show:
                fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
                for a, (im, t) in zip(ax, [(x, "real input"), (simulate_cvd_np(x, v), f"input @ {cvd}"),
                                           (o, "ours"), (simulate_cvd_np(o, v), f"ours @ {cvd}")]):
                    a.imshow(np.clip(im, 0, 1)); a.set_title(t); a.axis("off")
                plt.suptitle(os.path.basename(p)); plt.tight_layout(); plt.show()
                shown += 1
    print(f"真实图表 ({len(paths)} 张)：提取调色板后的平均 ΔS00")
    for cn in CVD_TYPES:
        print(f"  {cn:14s} {np.mean(stats[cn]):6.2f}  (n={len(stats[cn])})")


run_real("unet_full.pt")

## 20. 消融实验

| Run | 改什么 | 验证什么 |
|---|---|---|
| `full` | 全部 | 参照 |
| `noacc` | `w_acc=0`（只模仿 target） | 像素级无障碍 loss 有没有用 |
| `nokeep` | `w_keep=0` | 结构保持项有没有用（看 PSNR_keep 掉多少） |
| `nocond` | 模型看不到 CVD 类型 | CVD conditioning 有没有用 |
| `coloronly` | target 不加 marker/纹理（`USE_CUES=False` 重新生成数据到另一个目录） | multi-cue 是否比纯改色好（尤其全色盲） |

> 记得：所有 run 用同一个数据集 / 同一个 split（`coloronly` 除外，它的 target 不同但图和 split 一样，因为 seed 相同）。有时间的话每个 run 跑 3 个 seed（改 `seed=`）报告均值±标准差。

In [ ]:
ABLATIONS = {
    "noacc":  dict(w_acc=0.0,      out="unet_noacc.pt"),
    "nokeep": dict(w_keep=0.0,     out="unet_nokeep.pt"),
    "nocond": dict(no_cond=True,   out="unet_nocond.pt"),
}
abl_results = {"full": res_full}
for name, kw in ABLATIONS.items():
    print(f"\n=== training ablation: {name} ===")
    train_unet(**kw)
    abl_results[name] = evaluate_unet(kw["out"])
for name, rows in abl_results.items():
    summarize(rows, name)

In [ ]:
# ---- coloronly 消融：重新生成只改颜色的 target（同样的图、同样的 split）----
# 运行前确认磁盘空间够（和主数据集一样大）
RUN_COLORONLY = False
if RUN_COLORONLY:
    _main_out, _main_cues = OUT_DIR, USE_CUES
    OUT_DIR, USE_CUES = "/content/cvd_pairs_coloronly", False
    build_dataset(out_dir=OUT_DIR)
    train_unet(out="unet_coloronly.pt")
    abl_results["coloronly"] = evaluate_unet("unet_coloronly.pt")
    summarize(abl_results["coloronly"], "coloronly")
    OUT_DIR, USE_CUES = _main_out, _main_cues

## 21. 写报告时要注意的点

1. **Target 是优化器生成的，不是人工标注**：要说清楚 U-Net 在"模仿 + 改进"一个优化器。U-Net 的价值是**不需要原始数据、直接处理像素**，并且能加 marker/纹理；第 19 节的真实图表实验就是在验证这一点。
2. **评估要独立于 target 的生成方式**：训练只优化 CIE76，所以主表报告 CIEDE2000；PSNR_keep 衡量结构保持；最好再加一个 VLM 问答准确率（比如在 CVD 模拟后的图上问 FigureQA 的问题），这是 proposal 里承诺的 interpretability 指标。
3. **ΔS 只衡量颜色**：加了 marker/纹理的样本，ΔS 低估了它的好处（尤其全色盲）。这正是需要 VLM 问答或小规模人工评估的原因。
4. **三个层次的方法放在一起比较**，故事就完整了：
   - 固定调色板（Okabe–Ito / Petroff，Hungarian 匹配）
   - 调色板网络（旧 PaletteRecolorNet）和 per-chart 优化（第 8 节，上界）
   - 像素级 U-Net（本 notebook）
5. **CVD 模拟只是 proxy**，不是真实色盲用户的可读性证明（旧报告已经写了，继续保留）。
6. 局限：渲染风格单一（matplotlib）；图例名字被改成 "Series i"；饼图未支持；真实图表需要缩放到 256，文字会糊。